# Use Case 1: Retrieval-Augmented Generation (RAG)

**The Concept:** 
Standard Large Language Models (LLMs) hallucinate or lack access to your private data. RAG is an architecture where we first retrieve specific private knowledge relevant to a user's question, and then pass that knowledge to the LLM to ground its answer.

**The Architecture:** 
We inject our internal documents into a SochDB Vector Collection. When a user asks a question, we use **Hybrid Search**—combining semantic meaning (Vector Similarity) and exact text matching (BM25 Keyword Search)—to retrieve the absolute best context to format into the LLM prompt.

### Step 0: Install Packages & Setup Environment

If you want to run this code directly, you will need a `.env` file in this directory containing your Google Gemini credentials:

```env
GEMINI_API_KEY=your_key_here
```

In [7]:
!pip install sochdb openai python-dotenv

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

CHAT_MODEL = "gemini-3-flash-preview"
EMBEDDING_MODEL = "gemini-embedding-001"

def get_embedding(text):
    """Calls Gemini to create a vector embedding of the text."""
    response = client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return response.data[0].embedding

def get_completion(prompt):
    """Calls Gemini to generate a chat response."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database and Namespace
We open an embedded database and create an isolated namespace for our HR knowledge base.

In [13]:
from sochdb import Database, CollectionConfig, DistanceMetric

# Open an embedded database and create an isolated namespace
db = Database.open("./rag_ai_app_db")
ns = db.create_namespace("hr_department_kb")

### Step 2: Configure a Hybrid Search Collection
We configure a Vector Collection and enable the BM25 text index (`enable_hybrid_search`). Gemini's `gemini-embedding-001` creates vectors of **3072 dimensions**.

In [14]:
# Configure a collection with Hybrid Search enabled for optimal RAG retrieval
config = CollectionConfig(
    name="company_policies",
    dimension=3072,                # Dimension matching gemini-embedding-001
    metric=DistanceMetric.COSINE,
    enable_hybrid_search=True,     # Enables BM25 keyword search alongside vector search
    content_field="text"           # The metadata field to index for exact text searches
)
collection = ns.create_collection(config)

### Step 3: Ingest Knowledge Base
We insert some company policies into the collection by generating real embeddings for them.

In [15]:
texts = [
    "All employees get exactly exactly exactly 20 days of paid time off per year starting January 1st.",
    "The company strictly uses SochDB internally for all AI and vector search applications."
]

print("Generating embeddings via LLM API...")
embeddings = [get_embedding(t) for t in texts]

collection.upsert(
    ids=["policy_1", "policy_2"],
    embeddings=embeddings,
    metadatas=[
        {"text": texts[0], "topic": "PTO"},
        {"text": texts[1], "topic": "Tech"}
    ]
)
print("Knowledge base ingested!")

Generating embeddings via LLM API...
Knowledge base ingested!


### Step 4: Retrieve Context and Generate an Answer (RAG Phase)
When a user asks a question, we convert that question into an embedding, fetch the best matching policy using SochDB, and then feed that context directly into the ChatGPT prompt.

In [16]:
# The user's prompt
user_query = "I'm a new employee here. How much PTO do I get?"
user_query_vector = get_embedding(user_query)

# Step 4a) Use SochDB Hybrid Search to find the best context
results = collection.hybrid_search(
    vector=user_query_vector,       
    text_query="PTO",       
    k=1,                       
    alpha=0.5   # 0.5 balances vector and keyword validation equally
)

rag_context = results[0].metadata['text']
print(f"\n✅ Sourced context from DB: {rag_context}")

# Step 4b) Ask the LLM, grounding it entirely in the DB context
rag_prompt = f"""
You are an HR Assistant. Answer the user's question politely using ONLY the context provided.
Do not hallucinate.

Context: {rag_context}
User Question: {user_query}
"""

final_answer = get_completion(rag_prompt)
print(f"\n🤖 RAG LLM Output:\n{final_answer}")


✅ Sourced context from DB: All employees get exactly exactly exactly 20 days of paid time off per year starting January 1st.

🤖 RAG LLM Output:
All employees get exactly exactly exactly 20 days of paid time off per year starting January 1st.
